# Reproducibility smoke test: 500-person sample

This notebook runs `../validation/benchmark_script.py`, **unmodified, the exact script used for the published benchmark**, against the 500-person sample in `../sample/`.

**Purpose:** show that the code runs end to end on the real delivered schema (partitioned `visits/` and `truth/truth_quarterly/`, all 27 scenarios).

**Not the purpose:** reproduce the published metrics. After the five-year filter the sample leaves about 441 people for Task 1 with only a handful of dementia events, so the cross-validated AUC is dominated by noise. Task 2 is also small. Do not quote any number from this notebook. The published n = 10,000 results are in the README.

In [1]:
import subprocess, sys
r = subprocess.run([sys.executable, "../validation/benchmark_script.py", "--data", "../sample", "--out", "./smoke_test_results.parquet"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr[-2000:])
assert r.returncode == 0

     task                                  model                   metric       value                                                             extra
0   task1                                    all                 n_people  441.000000                                                          events 6
1   task1                                    all            event_rate_5y    0.013605                                                                  
2   task1     logistic_regression [all_features]                  roc_auc    0.654023                                  5-fold stratified, seed 20261108
3   task1     logistic_regression [all_features]                    brier    0.014323                                                                  
14  task1                 xgboost [all_features]                  roc_auc    0.653640                                  5-fold stratified, seed 20261108
15  task1                 xgboost [all_features]                    brier    0.014910   

## Reading the output

- Task 1 `n_people` and the `events` count in the `extra` column show how few dementia events the sample holds. With that few positives, an AUC from 5-fold CV moves a lot with the seed.
- Task 2 `true_mean_effect` and `true_effect_sd` come from the hidden truth tables, which is the feature of this dataset. The T-learner errors on 400 people are far noisier than on the full cohort.
- Published results on the full 10,000-person dataset: see the README.

In [2]:
import pandas as pd
res = pd.read_parquet("./smoke_test_results.parquet")
res[~res.metric.str.startswith("calibration")]

,task,model,metric,value,extra
0,task1,all,n_people,441.000000,events 6
1,task1,all,event_rate_5y,0.013605,
2,task1,logistic_regression [all_features],roc_auc,0.654023,"5-fold stratified, seed 20261108"
3,task1,logistic_regression [all_features],brier,0.014323,
14,task1,xgboost [all_features],roc_auc,0.653640,"5-fold stratified, seed 20261108"
15,task1,xgboost [all_features],brier,0.014910,
26,task1,logistic_regression [ptau217_only],roc_auc,0.849808,"5-fold stratified, seed 20261108"
27,task1,logistic_regression [ptau217_only],brier,0.013350,
28,task1,logistic_regression [ptau217_mmse_e4],roc_auc,0.769349,"5-fold stratified, seed 20261108"
29,task1,logistic_regression [ptau217_mmse_e4],brier,0.013678,
